# Topolojik Dinamik Sistemler

**Modül:** `pytop.dynamical_systems`  
**Kaynak:** Adams & Franzosa *Introduction to Topology* §8.1–8.2

---

Bir **topolojik dinamik sistem** $(X, f)$, bir topolojik uzay $X$ ile sürekli bir fonksiyon $f : X \to X$'in birleşimidir.  
Amaç, $f$'nin tekrarlı uygulamasının uzaydaki noktaları nasıl hareket ettirdiğini anlamaktır.

**Motivasyon:**  
- Hava durumu modelleri, nüfus dinamikleri, sarkacın hareketi — hepsi $(X, f)$ yapısına sahiptir.
- Topoloji, *ne kadar yakın başlarsa* orbitler o kadar yakın kalır mı? sorusunu yanıtlar.
- Kaos teorisi, küçük pertürbasyonların orbitler üzerindeki etkisini tam olarak topolojik dil ile ifade eder.

Bu notebook yedi bölümden oluşur:
1. **Orbit Kavramı** — Orbit tanımı, türler (sabit, periyodik, aperiyodik)
2. **Sabit Nokta Teorisi** — Brouwer ve Sharkovskii teoremleri, kararlılık sınıflandırması
3. **Topolojik Eşyapılılık** — Conjugacy, semi-conjugacy, dinamik invariantlar
4. **Kaos Kavramı** — Devaney tanımı, geçişlilik, hassas bağımlılık
5. **API** — `OrbitProfile`, `FixedPointProfile`, `TopologicalConjugacyProfile` fonksiyon tablosu
6. **Örnekler** — Dört senaryo: orbit tiplerinden kaotik sistemlere
7. **Alıştırmalar** — Kodlama, teori ve eşyapılılık analizi

## 1. Orbit Kavramı

### Tanım

$(X, f)$ bir dinamik sistem ve $x \in X$ olsun.  
$x$'in **orbit**i:
$$\mathcal{O}(x) = \{x,\, f(x),\, f^2(x),\, f^3(x),\, \ldots\}$$
şeklinde tanımlanır; burada $f^n = f \circ f \circ \cdots \circ f$ ($n$ kez).

### Orbit Türleri

| Tür | Koşul | Örnek |
|-----|-------|-------|
| **Sabit nokta** | $f(x) = x$ | $f(x)=x$ üzerinde her $x$ |
| **Periyodik** | $f^n(x) = x$, $f^k(x)\neq x$ ($k<n$) | $S^1$ üzerinde rasyonel döndürme |
| **Sonunda periyodik** | $f^{m+n}(x) = f^m(x)$ bazı $m,n \geq 1$ için | $f(x)=x^2$ üzerinde $(0,1)$'deki noktalar |
| **Sonsuz aperiyodik** | $f^n(x) \neq x$ tüm $n \geq 1$ için | $S^1$ üzerinde irrasyonel döndürme |

In [ ]:
from pytop.dynamical_systems import get_orbit_profiles, orbit_type_summary

profiles = get_orbit_profiles()
print(f"Orbit profil sayısı: {len(profiles)}\n")

for p in profiles:
    print(f"── {p.display_name}")
    print(f"   Tip     : {p.orbit_type}")
    print(f"   Periyot : {p.period}")
    print(f"   Uzay    : {p.space_description}")
    print(f"   Kaynak  : {p.source_section}")
    print()

In [ ]:
summary = orbit_type_summary()
print("Orbit tipi → profil anahtarları:")
for orbit_type, keys in summary.items():
    print(f"  {orbit_type:25s}: {keys}")

## 2. Sabit Nokta Teorisi

### Tanım

$f(x_0) = x_0$ koşulunu sağlayan $x_0 \in X$'e **sabit nokta** denir.  
Sabit noktalar periyodik orbitlerinin en özel durumudur (periyot 1).

### Kararlılık Sınıflandırması

$(X, d)$ metrik uzayında $f(x_0) = x_0$ olsun:

- **Çekici (stable):** Bir $U \ni x_0$ komşuluğu vardır, $x \in U \Rightarrow f^n(x) \to x_0$.
- **İtici (unstable):** Her komşuluk $U \ni x_0$ için bir $x \in U$ ve $n$ ile $f^n(x) \notin U$.
- **Nötr:** Yakın orbitler yakında kalır ama yakınsamaz.

### Teorem (Brouwer Sabit Nokta Teoremi)

> **Teorem.** $f : D^n \to D^n$ sürekli ise $f$'nin en az bir sabit noktası vardır.

**İspat fikri (n=2):**  
Sabit noktanın olmadığını varsayalım. Her $x \in D^2$ için $f(x) \neq x$; bu durumda $x$'ten $f(x)$'e çizilen ışının $\partial D^2$ ile kesişim noktasını $r(x)$ olarak tanımla. Bu $r : D^2 \to S^1$ sürekli bir geri çekilme (retraction) tanımlar. Ancak $S^1$ homotopi teorisi gereği $D^2$'nin geri çekilme retraksiyonu olamaz — çelişki. $\square$

### Teorem (Sharkovskii)

> **Teorem.** $f : \mathbb{R} \to \mathbb{R}$ sürekli ve periyot-3 bir orbit içeriyorsa, $f$'nin her $n \geq 1$ periyotlu orbitler içerir.

Sharkovskii sıralaması: $3 \triangleright 5 \triangleright 7 \triangleright \cdots \triangleright 2{\cdot}3 \triangleright 2{\cdot}5 \triangleright \cdots \triangleright 4 \triangleright 2 \triangleright 1$.

In [ ]:
from pytop.dynamical_systems import get_fixed_point_profiles, fixed_point_stability_summary

fp_profiles = get_fixed_point_profiles()
print(f"Sabit nokta profil sayısı: {len(fp_profiles)}\n")

for p in fp_profiles:
    print(f"── {p.display_name}")
    print(f"   Kararlılık        : {p.stability}")
    print(f"   Varlık teoremi    : {p.existence_theorem}")
    print(f"   Uzay              : {p.space_description}")
    print()

In [ ]:
stab_summary = fixed_point_stability_summary()
print("Kararlılık sınıfı → profil anahtarları:")
for stab, keys in stab_summary.items():
    print(f"  {stab:20s}: {keys}")

## 3. Topolojik Eşyapılılık (Conjugacy)

### Tanım

$(X, f)$ ve $(Y, g)$ dinamik sistemleri **topolojik olarak eşyapılı** (topologically conjugate) olmak için $h : X \to Y$ homeomorfizması:
$$h \circ f = g \circ h$$
koşulunu sağlamalıdır.

Eşyapılılık, dinamik sistemler için bir izomorfizm kavramıdır: $h$ altında $f$-orbitler $g$-orbitlere birebir karşılık gelir.

### Eşyapılılık İnvaryantları

Aşağıdaki nicelikler topolojik eşyapılılık altında korunur:

| İnvaryant | Açıklama |
|-----------|----------|
| Periyodik orbit kümesi | Periyot-$n$ orbitlerinin sayısı aynı kalır |
| Topolojik entropi | $h_{top}(f) = h_{top}(g)$ |
| Geçişlilik (transitivity) | $f$ topolojik transitif $\Leftrightarrow$ $g$ transitif |
| Hassas bağımlılık | $f$ kaotik $\Leftrightarrow$ $g$ kaotik |

### Yarı-Eşyapılılık (Semi-conjugacy)

$h : X \to Y$ **sürekli ve örten** (surjektif) ise, $h \circ f = g \circ h$ koşulunu sağlayan $h$'ye yarı-eşyapılılık (factor map) denir.  
$(Y, g)$, $(X, f)$'nin bir **faktörüdür**. Entropi faktör haritası altında artmaz: $h_{top}(g) \leq h_{top}(f)$.

In [ ]:
from pytop.dynamical_systems import get_conjugacy_profiles, conjugacy_exists_summary

conj_profiles = get_conjugacy_profiles()
print(f"Eşyapılılık profil sayısı: {len(conj_profiles)}\n")

for p in conj_profiles:
    print(f"── {p.display_name}")
    print(f"   Eşyapılılık var mı: {p.conjugacy_exists}")
    print(f"   Sistem f          : {p.system_f}")
    print(f"   Sistem g          : {p.system_g}")
    if p.invariants_preserved:
        print(f"   Korunan inv.      : {', '.join(p.invariants_preserved[:2])}")
    print()

## 4. Kaos Kavramı

Bir dinamik sistem $(X, f)$ **topolojik olarak kaotik** (Devaney anlamında) sayılır, eğer:

1. **Topolojik geçişlilik:** Her iki boş olmayan açık $U, V \subseteq X$ için bir $n \geq 0$ ile $f^n(U) \cap V \neq \emptyset$.
2. **Yoğun periyodik nokta kümesi:** $\{x \in X : f^n(x) = x \text{ bazı } n \geq 1 \}$, $X$'te yoğun.
3. **Başlangıç koşullarına hassas bağımlılık:** Bir $\delta > 0$ vardır; her $x \in X$ ve $\varepsilon > 0$ için bir $y$ ile $d(x,y) < \varepsilon$ ve $d(f^n(x), f^n(y)) > \delta$ olan $n$ bulunur.

> **Not:** Banks ve ark. (1992) gösterdi ki koşullar 1 ve 2, koşul 3'ü *otomatik olarak* gerektirir.  
> Yani kaos için geçişlilik + yoğun periyodik noktalar yeterlidir.

**Standart kaotik örnekler:**
- İkiye katlama haritası $d : S^1 \to S^1$, $d(\theta) = 2\theta \mod 2\pi$
- Çadır haritası $T : [0,1] \to [0,1]$, $T(x) = \min(2x, 2-2x)$
- Lojistik harita $f_4 : [0,1] \to [0,1]$, $f_4(x) = 4x(1-x)$

## 5. API Tablosu

| Fonksiyon / Sınıf | Açıklama | Döndürür |
|-------------------|----------|----------|
| `OrbitProfile` | Orbit tipini tanımlayan profil | `dataclass(frozen=True)` |
| `get_orbit_profiles()` | Tüm orbit profillerini döndürür | `tuple[OrbitProfile, ...]` |
| `orbit_type_summary()` | Orbit tipini anahtar listesine eşler | `dict[str, list[str]]` |
| `FixedPointProfile` | Sabit nokta kararlılık profili | `dataclass(frozen=True)` |
| `get_fixed_point_profiles()` | Tüm sabit nokta profillerini döndürür | `tuple[FixedPointProfile, ...]` |
| `fixed_point_stability_summary()` | Kararlılık sınıfını anahtar listesine eşler | `dict[str, list[str]]` |
| `TopologicalConjugacyProfile` | Eşyapılılık profili | `dataclass(frozen=True)` |
| `get_conjugacy_profiles()` | Tüm eşyapılılık profillerini döndürür | `tuple[TopologicalConjugacyProfile, ...]` |
| `conjugacy_exists_summary()` | Eşyapılılık varlığını anahtar listesine eşler | `dict[bool, list[str]]` |
| `dynamical_systems_profile_registry()` | Kategori başına profil sayısı | `dict[str, int]` |

## 6. Örnekler

In [ ]:
# Örnek 1: Orbit tiplerini ayrıştır ve açıklamalarını yazdır
from pytop.dynamical_systems import get_orbit_profiles

print("=== Orbit Tipleri ve Matematiksel Açıklamalar ===")
for p in get_orbit_profiles():
    print(f"\n{p.display_name} [{p.orbit_type}]")
    # İlk 120 karakter (kısa özet)
    snippet = p.notes[:120].rstrip()
    print(f"  → {snippet}...")

In [ ]:
# Örnek 2: Çekici sabit noktanın Banach teoremi ile ilişkisi
from pytop.dynamical_systems import get_fixed_point_profiles

banach_related = [
    p for p in get_fixed_point_profiles()
    if "Banach" in p.existence_theorem or "Brouwer" in p.existence_theorem
]

print("Varlık teoremi olan sabit nokta profilleri:")
for p in banach_related:
    print(f"  {p.display_name}")
    print(f"    Teorem : {p.existence_theorem}")
    print(f"    Uzay   : {p.space_description}")

In [ ]:
# Örnek 3: Eşyapılı ve eşyapılı olmayan sistemleri ayırt et
from pytop.dynamical_systems import get_conjugacy_profiles

conjugate     = [p for p in get_conjugacy_profiles() if p.conjugacy_exists]
non_conjugate = [p for p in get_conjugacy_profiles() if not p.conjugacy_exists]

print("Topolojik olarak eşyapılı sistemler:")
for p in conjugate:
    print(f"  ✓ {p.display_name}")
    print(f"      f : {p.system_f}")
    print(f"      g : {p.system_g}")

print("\nEşyapılı OLMAYAN sistemler:")
for p in non_conjugate:
    print(f"  ✗ {p.display_name}")
    print(f"      Neden: eşyapılılık homeomorfizması sabit noktaları korur")

In [ ]:
# Örnek 4: Kayıt defteri özeti
from pytop.dynamical_systems import dynamical_systems_profile_registry

registry = dynamical_systems_profile_registry()
total = sum(registry.values())

print("Dinamik Sistemler Profil Kaydı")
print("─" * 40)
for category, count in registry.items():
    print(f"  {category:35s}: {count}")
print("─" * 40)
print(f"  {'Toplam':35s}: {total}")

## 7. Alıştırmalar

**Alıştırma 1.**  
$f : \mathbb{R} \to \mathbb{R}$, $f(x) = \cos(x)$ fonksiyonunu ele alın.  
Sabit noktalar $f(x) = x$ denkleminin çözümleridir (Dottie sayısı $\approx 0.739$).  
Bu sabit noktanın çekici mi, itici mi yoksa nötr mü olduğunu kararlılık tanımını kullanarak gösterin.

**Alıştırma 2.**  
$f : \{1,2,3,4\} \to \{1,2,3,4\}$ (ayrık topoloji), $f = (1\,2)(3)(4)$ (döngü gösterimi) fonksiyonu için:
(a) Her noktanın orbitini belirleyin.  
(b) Sabit nokta sayısını bulun.  
(c) $g : \{1,2,3,4\} \to \{1,2,3,4\}$, $g = (1\,2\,3)(4)$ ile eşyapılı olup olmadığını sabit nokta sayısı invariantını kullanarak karar verin.

**Alıştırma 3.**  
$S^1$ üzerinde $\alpha$ açısı kadar döndürme fonksiyonu $R_\alpha$ düşünün.  
(a) $\alpha$ rasyonel ise tüm orbitler neden periyodiktir?  
(b) $\alpha$ irrasyonel ise her orbitin $S^1$'de yoğun olduğunu Weyl eşdağılım teoremi çerçevesinde açıklayın.  
(c) Bu iki durum topolojik olarak eşyapılı mıdır? (İpucu: periyodik orbit kümesi invariantı.)

**Alıştırma 4.**  
İkiye katlama haritası $d : S^1 \to S^1$, $d(\theta) = 2\theta \mod 2\pi$:  
(a) $d$'nin periyot-$n$ noktalarının sayısının $2^n - 1$ olduğunu doğrulayın.  
(b) Çadır haritası $T : [0,1] \to [0,1]$ ile $d$'nin topolojik olarak eşyapılı olduğunu belirten `TopologicalConjugacyProfile`'i `get_conjugacy_profiles()` çıktısından bulun ve korunan invariantları listeleyin.  
(c) Bu sistemlerin her ikisinin de kaotik olduğunu eşyapılılık invariantları üzerinden tartışın.

---

## Özet

| Kavram | Ana Sonuç |
|--------|-----------|
| Orbit | Her nokta sabit, periyodik, sonunda periyodik veya sonsuz aperiyodik |
| Sabit nokta kararlılığı | Çekici, itici, nötr |
| Brouwer teoremi | $D^n \to D^n$ sürekli ⟹ sabit nokta var |
| Sharkovskii teoremi | Periyot-3 ⟹ tüm periyotlar mevcuttur |
| Eşyapılılık | Periyodik orbit yapısını, entropiyi, kaotikliği korur |
| Yarı-eşyapılılık | Entropi azalabilir ama artamaz |

**Bir sonraki adım:** `pytop.degree_theory` — Brouwer derecesi ve sarım sayısı.